# 《深山传家》Colab 批量 2.5D 烘培 + 明清木刻工笔风格化

用法：
1. 云盘 `shenshan_td/raw_models/` 里放好 `.glb`（由 `colab_gen_3d.ipynb` 生成，文件名 = `assets_manifest.json` 的 `id`）。
2. 菜单「代码执行程序 → 更改运行时类型」选 **GPU（T4 / L4 / A100 均可，无需 High-RAM）**。
3. `SYNC_FROM_GITHUB=True` 会先从公开仓库拉最新的 manifest、烘培脚本、风格化脚本与 `style_profile.json` 到云盘。
4. 先把 `TEST_SINGLE` 设为单个文件名试看：输出到 `baked_sprites/_preview/` 并强制覆盖，不会挡住之后的正式批量烘培；满意后留空，`SCALE=1.0, SAMPLES=48` 正式烘培全部模型到 `baked_sprites/`。
5. 只改了 `style_profile.json` 想重出图、不想重新渲染：设 `RESTYLE_ONLY=True`（几秒一件）。
6. 最后一格自动释放运行时。
7. 5 件程序化资产（稻田、篱笆、青砖墙、捕兽夹）不走图生 3D：第 ④ 格开头会自动用 `make_proxy_assets.py` 生成到 `raw_models/`。
8. `PROXY_FALLBACK=True`：还没有 AI 模型的建筑 / 道具用程序化替身烘到 `baked_sprites/_proxy/`。导出时先导 `_proxy/`、再导 `baked_sprites/`（`export_sprites.py --merge`），同 id 的 AI 版会覆盖替身。

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/shenshan_td"
SYNC_FROM_GITHUB = True    # 先从公开仓库同步最新脚本与画风参数到云盘
GITHUB_RAW = "https://raw.githubusercontent.com/dongdongw12/shenshan_td_project/main"
BLENDER = "5.3.0"          # 与本机 Blender 版本一致（脚本也兼容 5.0.x）
TEST_SINGLE = "house_lv2_tile.glb"  # 填文件名只试烘 1 个（输出到 _preview/）；留空 "" 正式烘培全部
MODE = "stylized"          # "stylized" = 工笔风格化（默认）| "classic" = v1 直出（仅对比用）
SCALE = 0.5                # stylized：比例倍率，0.5 预览 / 1.0 正式（128 px/m）
SAMPLES = 24               # Cycles 采样数（正式批量建议 48）
DEVICE = "auto"            # OPTIX 报错则改为 "CUDA"
OVERWRITE = False          # True 会重烘已有文件（TEST_SINGLE 模式总是覆盖）
RESTYLE_ONLY = False       # True = 不渲染，只用已有 _passes 重新风格化
BAKE_CHARACTERS = True    # 同时烘角色序列帧到 baked_sprites/chars/：9 个敌人（行走 + 攻击）+ 3 个家人（行走 + 攻击 + 干活）
CHARACTERS = "wolf,boar,bear_king,scout,swordsman,shield,arsonist,second_chief,chief,hero,wife,child"
PROXY_FALLBACK = True     # raw_models/ 里还没有 AI 模型的建筑，用程序化替身烘到 baked_sprites/_proxy/（过渡美术 / 图生 3D 失败时的兜底）

In [ ]:
import os, subprocess, urllib.request
from google.colab import drive
drive.mount('/content/drive')

SYNC = {"assets_manifest.json": "assets_manifest.json",
        "pipeline/blender/bake_isometric_sprites.py": "bake_isometric_sprites.py",
        "pipeline/style/stylize_sprites.py": "stylize_sprites.py",
        "pipeline/style/style_profile.json": "style_profile.json",
        "pipeline/blender/make_proxy_assets.py": "make_proxy_assets.py",
        "pipeline/blender/make_characters.py": "make_characters.py",
        "pipeline/blender/bake_sprite_anim.py": "bake_sprite_anim.py"}
if SYNC_FROM_GITHUB:
    for src, dst in SYNC.items():
        urllib.request.urlretrieve(f"{GITHUB_RAW}/{src}", f"{DRIVE_DIR}/{dst}")
    print("已从 GitHub 同步：", ", ".join(SYNC.values()))
for dst in SYNC.values():
    assert os.path.exists(f"{DRIVE_DIR}/{dst}"), f"云盘缺少 {dst}"
assert os.path.exists(f"{DRIVE_DIR}/raw_models"), "请先建立 raw_models 目录并放入 .glb 模型"
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)

mm = ".".join(BLENDER.split(".")[:2])
if not os.path.exists("/content/blender/blender"):
    !apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libxkbcommon0 libsm6 libgl1 libegl1 > /dev/null
    !wget -q https://download.blender.org/release/Blender{mm}/blender-{BLENDER}-linux-x64.tar.xz -O /content/blender.tar.xz
    !mkdir -p /content/blender && tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1
!/content/blender/blender --version | head -1
has_oiio = "OK" in subprocess.run(["/content/blender/blender", "-b", "--python-expr", "import OpenImageIO; print('OK')"],
                                  capture_output=True, text=True).stdout
if not has_oiio:
    !pip install -q OpenEXR pillow
print("风格化在 Blender 内运行" if has_oiio else "风格化改用 Colab Python 运行")

In [ ]:
# 程序化资产（稻田青苗/金穗、篱笆、青砖墙、捕兽夹）：云盘里还没有就先用 Blender 生成 .glb
PROCEDURAL = ["farm_rice_young", "farm_rice_ripe", "fence_wood", "wall_brick", "trap_bear"]
todo_proc = [i for i in PROCEDURAL if not os.path.exists(f"{DRIVE_DIR}/raw_models/{i}.glb")]
if todo_proc and not RESTYLE_ONLY:
    !/content/blender/blender -b -P "{DRIVE_DIR}/make_proxy_assets.py" -- --out "{DRIVE_DIR}/raw_models" --ids {",".join(todo_proc)} 2>&1 | grep -E "^PROXY|Error|Traceback"

# 替身兜底：AI 模型还没有的件先用程序化替身烘出来（单独目录，AI 模型出来后正式烘培不会被它挡住）
PROXIES = ["house_lv1_thatch", "house_lv2_tile", "house_lv3_courtyard", "house_wing_room", "hunt_lodge", "granary",
           "ancestral_hall", "tower_bamboo", "tower_crossbow", "tower_oil_fire", "tower_rolling_log", "fish_weir",
           "prop_peddler_cart", "prop_stone_lantern", "prop_beacon_brazier", "prop_pine_tree"]
if PROXY_FALLBACK and not TEST_SINGLE and not RESTYLE_ONLY:
    missing = [i for i in PROXIES if not os.path.exists(f"{DRIVE_DIR}/raw_models/{i}.glb")]
    if missing:
        print("用替身兜底：", ", ".join(missing))
        !/content/blender/blender -b -P "{DRIVE_DIR}/make_proxy_assets.py" -- --out "{DRIVE_DIR}/proxy_models" --ids {",".join(missing)} 2>&1 | grep -E "^PROXY|Error|Traceback"
        popts = f"--mode stylized --scale {SCALE} --samples {SAMPLES} --device {DEVICE} --profile '{DRIVE_DIR}/style_profile.json'" + ("" if has_oiio else " --skip-stylize")
        !/content/blender/blender -b -P "{DRIVE_DIR}/bake_isometric_sprites.py" -- --input "{DRIVE_DIR}/proxy_models" --out "{DRIVE_DIR}/baked_sprites/_proxy" --manifest "{DRIVE_DIR}/assets_manifest.json" {popts} 2>&1 | grep -E "^BAKED|^SKIP|^ALL_|Error|Traceback"

in_target = f"{DRIVE_DIR}/raw_models/{TEST_SINGLE}" if TEST_SINGLE else f"{DRIVE_DIR}/raw_models"
out_dir = f"{DRIVE_DIR}/baked_sprites/_preview" if TEST_SINGLE else f"{DRIVE_DIR}/baked_sprites"
profile = f"{DRIVE_DIR}/style_profile.json"
only = os.path.splitext(TEST_SINGLE)[0] if TEST_SINGLE else ""

if not RESTYLE_ONLY:
    opts = f"--mode {MODE} --scale {SCALE} --samples {SAMPLES} --device {DEVICE} --profile '{profile}'"
    opts += " --overwrite" if (OVERWRITE or TEST_SINGLE) else ""
    opts += "" if (has_oiio or MODE == "classic") else " --skip-stylize"
    !/content/blender/blender -b -P "{DRIVE_DIR}/bake_isometric_sprites.py" -- --input "{in_target}" --out "{out_dir}" --manifest "{DRIVE_DIR}/assets_manifest.json" {opts} 2>&1 | grep -E "^BAKE|^START|^RENDERED|^BAKED|^SKIP|^ALL_|Error|Traceback"
if MODE == "stylized" and (RESTYLE_ONLY or not has_oiio):
    py = f"/content/blender/{mm}/python/bin/python3*" if has_oiio else "python"
    !{py} "{DRIVE_DIR}/stylize_sprites.py" --passes "{out_dir}/_passes" --out "{out_dir}" --profile "{profile}" --only "{only}"

# 角色序列帧（与建筑同一套风格化；夜晚版由白昼调色）
if BAKE_CHARACTERS and MODE == "stylized" and not RESTYLE_ONLY and has_oiio:
    chars_out = f"{out_dir}/chars"
    !/content/blender/blender -b -P "{DRIVE_DIR}/bake_sprite_anim.py" -- --chars {CHARACTERS} --out "{chars_out}" --scale {SCALE} --samples {SAMPLES} --profile "{profile}" 2>&1 | grep -E "^BAKED_ANIM|^ALL_ANIM|Error|Traceback"

In [ ]:
# 烘培完毕自动释放 Colab GPU 运行时，防止空转扣费
from google.colab import runtime
runtime.unassign()